# Medical History data analysis (Snowflake, read-only)

One row = one recorded medical history condition. **Read-only**: `SELECT` / `DESCRIBE` only.

**Cell format:** analysis cells are **SQL cells**, so each result shows the native grid with **Table / Chart / Pivot** and the **download** button.

**Config stays in one place.** The Python cell below builds the table and column names; SQL cells read them with Jinja braces, for example `{{T}}` and `{{C_SNOMED}}`.

**Grain:** `MedicalHistoryId` should be unique. One patient can have many conditions.

**SNOMED focus:** the same code can be written with different `Value` spellings. Section 9 counts unique codes, then name variations per code.

## 1. Active session

In [ ]:
import pandas as pd

from snowflake.snowpark.context import get_active_session

session = get_active_session()
session

## 2. Config (change names ONLY here)

SQL cells can only read **plain string** variables, so this cell also exposes flat names (`T`, `C_VAL`, `C_SNOMED`, ...). `{{COL['snomed']}}` would not work in a SQL cell.

Set `QUOTE_COLUMNS = False` if your table uses unquoted uppercase names.

In [ ]:
DATABASE_NAME = "ATTR"
SCHEMA_NAME = "PUBLIC"
TABLE_NAME = "MEDICAL_HISTORY"   # try MEDICALHISTORY, MEDICALHX, MED_HISTORY

QUOTE_DATABASE = False
QUOTE_SCHEMA = False
QUOTE_TABLE = False
QUOTE_COLUMNS = True

COL = {
    "medical_history_id": "MedicalHistoryId",
    "encounter_id": "EncounterId/VisitId",
    "patient_id": "Member/PatientId",
    "source_category": "Source/Category",
    "value": "Value",
    "snomed": "SNOMED",
    "secondary_snomed": "Secondary SNOMED",
    "date": "Date",
}


def sf_ident(name, quoted):
    if quoted:
        return '"' + str(name).replace('"', '""') + '"'
    return str(name)


def col(key):
    return sf_ident(COL[key], QUOTE_COLUMNS)


# Flat strings for SQL cells
DB = sf_ident(DATABASE_NAME, QUOTE_DATABASE)
T = ".".join(
    [
        DB,
        sf_ident(SCHEMA_NAME, QUOTE_SCHEMA),
        sf_ident(TABLE_NAME, QUOTE_TABLE),
    ]
)

C_ID = col("medical_history_id")
C_ENC = col("encounter_id")
C_PT = col("patient_id")
C_SRC = col("source_category")
C_VAL = col("value")
C_SNOMED = col("snomed")
C_SNOMED2 = col("secondary_snomed")
C_DATE = col("date")

for name, value in [
    ("DB", DB),
    ("T", T),
    ("C_ID", C_ID),
    ("C_ENC", C_ENC),
    ("C_PT", C_PT),
    ("C_SRC", C_SRC),
    ("C_VAL", C_VAL),
    ("C_SNOMED", C_SNOMED),
    ("C_SNOMED2", C_SNOMED2),
    ("C_DATE", C_DATE),
]:
    print(f"{name} = {value}")

## 3. Find the table (only if the name or schema is wrong)

In [ ]:
SELECT
    CURRENT_ROLE() AS ROLE,
    CURRENT_WAREHOUSE() AS WAREHOUSE,
    CURRENT_DATABASE() AS DATABASE,
    CURRENT_SCHEMA() AS SCHEMA;

In [ ]:
SELECT
    TABLE_CATALOG,
    TABLE_SCHEMA,
    TABLE_NAME,
    ROW_COUNT,
    BYTES
FROM {{DB}}.INFORMATION_SCHEMA.TABLES
WHERE TABLE_TYPE = 'BASE TABLE'
  AND (
        UPPER(TABLE_NAME) LIKE '%MEDICAL%'
     OR UPPER(TABLE_NAME) LIKE '%HISTORY%'
     OR UPPER(TABLE_NAME) LIKE '%CONDITION%'
  )
ORDER BY TABLE_SCHEMA, TABLE_NAME;

## 4. Table shape and first 10 rows

If a later cell fails with **invalid identifier**, copy the exact names from this `DESCRIBE` result into `COL` in the config cell.

In [ ]:
DESCRIBE TABLE {{T}};

In [ ]:
SELECT *
FROM {{T}}
LIMIT 10;

## 5. Volume and uniqueness

Expected: unique `MedicalHistoryId` close to `ROW_COUNT`. Fewer unique patients means people have several conditions.

In [ ]:
SELECT
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_ID}}) AS UNIQUE_MEDICAL_HISTORY_IDS,
    COUNT(DISTINCT {{C_ENC}}) AS UNIQUE_ENCOUNTERS,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_SRC}}) AS UNIQUE_SOURCE_CATEGORIES,
    COUNT(DISTINCT {{C_VAL}}) AS UNIQUE_CONDITION_VALUES,
    COUNT(DISTINCT {{C_SNOMED}}) AS UNIQUE_SNOMED_CODES,
    COUNT(*) - COUNT(DISTINCT {{C_ID}}) AS EXTRA_ROWS_VS_UNIQUE_ID,
    ROUND(COUNT(*) / NULLIF(COUNT(DISTINCT {{C_PT}}), 0), 2) AS AVG_CONDITIONS_PER_PATIENT
FROM {{T}};

## 6. Completeness (nulls)

Required per the dictionary: id, encounter, patient, source/category, value, date. Optional: SNOMED, Secondary SNOMED.

In [ ]:
SELECT
    COUNT(*) AS ROW_COUNT,
    SUM(IFF({{C_ID}} IS NULL, 1, 0)) AS NULL_MEDICAL_HISTORY_ID,
    SUM(IFF({{C_ENC}} IS NULL, 1, 0)) AS NULL_ENCOUNTER_ID,
    SUM(IFF({{C_PT}} IS NULL, 1, 0)) AS NULL_PATIENT_ID,
    SUM(IFF({{C_SRC}} IS NULL, 1, 0)) AS NULL_SOURCE_CATEGORY,
    SUM(IFF({{C_VAL}} IS NULL, 1, 0)) AS NULL_VALUE,
    SUM(IFF({{C_SNOMED}} IS NULL, 1, 0)) AS NULL_SNOMED,
    SUM(IFF({{C_SNOMED2}} IS NULL, 1, 0)) AS NULL_SECONDARY_SNOMED,
    SUM(IFF({{C_DATE}} IS NULL, 1, 0)) AS NULL_DATE
FROM {{T}};

## 7. Source/Category — unique values and counts

Text field, so we group by the stored string (examples: Skin Disease History, Past Medical History).

`CONDITION_LIST` spells out the condition names in each category. If a category holds thousands of names, `LISTAGG` can hit its 16 MB limit — in that case delete that one line and use `source_by_value` in section 8, which lists the same thing one row per pair.

In [ ]:
SELECT
    {{C_SRC}} AS SOURCE_CATEGORY,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_VAL}}) AS UNIQUE_CONDITIONS,
    LISTAGG(DISTINCT {{C_VAL}}::STRING, ' | ') AS CONDITION_LIST,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM {{T}}
GROUP BY 1
ORDER BY ROW_COUNT DESC;

## 8. Value - unique conditions, occurrences, and their SNOMED codes

`Value` is the condition name (text). Group by the stored string; dictionary examples (`Hypercholesterolemia`, `Depression`, `Anxiety`, `Diabetes`, `Basal Cell Skin Cancer`) are only examples and are not hardcoded.

`value_counts` is the **full list**: one row per unique condition, with `ROW_COUNT` (occurrences) and the actual `SNOMED_CODES` / `SECONDARY_SNOMED_CODES` attached to that name.

In [ ]:
SELECT
    {{C_VAL}} AS CONDITION_VALUE,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_ENC}}) AS UNIQUE_ENCOUNTERS,
    COUNT(DISTINCT {{C_SNOMED}}) AS DISTINCT_SNOMED_CODES,
    LISTAGG(DISTINCT {{C_SNOMED}}::STRING, ' | ') AS SNOMED_CODES,
    COUNT(DISTINCT {{C_SNOMED2}}) AS DISTINCT_SECONDARY_SNOMED_CODES,
    LISTAGG(DISTINCT {{C_SNOMED2}}::STRING, ' | ') AS SECONDARY_SNOMED_CODES,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM {{T}}
GROUP BY 1
ORDER BY ROW_COUNT DESC, CONDITION_VALUE;

In [ ]:
SELECT
    {{C_SRC}} AS SOURCE_CATEGORY,
    {{C_VAL}} AS CONDITION_VALUE,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM {{T}}
GROUP BY 1, 2
ORDER BY ROW_COUNT DESC;

### Word-level counts inside Value

`Basal Cell Skin Cancer` is one condition name, but the word `Skin` also appears in other names. This splits `Value` on punctuation and spaces and counts each word.

`WORD_OCCURRENCES` = how many rows contain that word.

In [ ]:
WITH tokens AS (
    SELECT
        {{C_PT}} AS PATIENT_ID,
        {{C_ENC}} AS ENCOUNTER_ID,
        TRIM(f.VALUE::STRING) AS WORD
    FROM {{T}},
         LATERAL FLATTEN(
             INPUT => SPLIT(
                 TRIM(REGEXP_REPLACE({{C_VAL}}::STRING, '[^A-Za-z0-9]+', ' ')),
                 ' '
             )
         ) f
    WHERE {{C_VAL}} IS NOT NULL
)
SELECT
    WORD,
    COUNT(*) AS WORD_OCCURRENCES,
    COUNT(DISTINCT PATIENT_ID) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT ENCOUNTER_ID) AS UNIQUE_ENCOUNTERS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_WORD_OCCURRENCES
FROM tokens
WHERE WORD IS NOT NULL
  AND WORD <> ''
GROUP BY 1
ORDER BY WORD_OCCURRENCES DESC, WORD;

## 9. SNOMED — unique codes and name variations

Six views:

1. `snomed_unique_counts` — how many unique codes exist and how many rows carry one.
2. `snomed_name_variations` — **per code**, how many different `Value` spellings, with the names listed in one column.
3. `snomed_value_pairs` — one row per code + name. `NAMES_FOR_THIS_CODE` says how many names that code has (4 means four different spellings), and `NAME_NUMBER` counts them 1, 2, 3, 4.
4. `value_with_many_snomed` — the reverse check: one name mapped to several codes.
5. `secondary_snomed_values` — the **unique Secondary SNOMED list** with occurrences and the condition names attached to each.
6. `snomed_pair_combinations` — which primary + secondary code pairs actually occur together.

Spellings are compared exactly as stored. `UNIQUE_NAMES_NORMALIZED` also shows the count after trimming, upper-casing, and collapsing spaces, so you can tell real naming differences from formatting noise.

In [ ]:
SELECT
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT NULLIF(TRIM({{C_SNOMED}}::STRING), '')) AS UNIQUE_SNOMED_CODES,
    COUNT(DISTINCT NULLIF(TRIM({{C_SNOMED2}}::STRING), '')) AS UNIQUE_SECONDARY_SNOMED_CODES,
    SUM(IFF({{C_SNOMED}} IS NOT NULL AND TRIM({{C_SNOMED}}::STRING) <> '', 1, 0)) AS ROWS_WITH_SNOMED,
    SUM(IFF({{C_SNOMED}} IS NULL OR TRIM({{C_SNOMED}}::STRING) = '', 1, 0)) AS ROWS_MISSING_SNOMED,
    ROUND(
        100.0 * SUM(IFF({{C_SNOMED}} IS NOT NULL AND TRIM({{C_SNOMED}}::STRING) <> '', 1, 0))
        / NULLIF(COUNT(*), 0),
        2
    ) AS PCT_ROWS_WITH_SNOMED,
    SUM(IFF({{C_SNOMED2}} IS NOT NULL AND TRIM({{C_SNOMED2}}::STRING) <> '', 1, 0)) AS ROWS_WITH_SECONDARY_SNOMED
FROM {{T}};

In [ ]:
SELECT
    {{C_SNOMED}} AS SNOMED,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_VAL}}) AS UNIQUE_NAMES,
    COUNT(DISTINCT UPPER(REGEXP_REPLACE(TRIM({{C_VAL}}::STRING), '\\s+', ' '))) AS UNIQUE_NAMES_NORMALIZED,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    LISTAGG(DISTINCT {{C_VAL}}::STRING, ' | ') AS NAME_VARIATIONS
FROM {{T}}
WHERE {{C_SNOMED}} IS NOT NULL
  AND TRIM({{C_SNOMED}}::STRING) <> ''
GROUP BY 1
ORDER BY UNIQUE_NAMES DESC, ROW_COUNT DESC;

In [ ]:
SELECT
    {{C_SNOMED}} AS SNOMED,
    {{C_VAL}} AS CONDITION_VALUE,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(*) OVER (PARTITION BY {{C_SNOMED}}) AS NAMES_FOR_THIS_CODE,
    ROW_NUMBER() OVER (PARTITION BY {{C_SNOMED}} ORDER BY COUNT(*) DESC) AS NAME_NUMBER,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY {{C_SNOMED}}), 2) AS PCT_WITHIN_CODE
FROM {{T}}
WHERE {{C_SNOMED}} IS NOT NULL
  AND TRIM({{C_SNOMED}}::STRING) <> ''
GROUP BY 1, 2
ORDER BY NAMES_FOR_THIS_CODE DESC, SNOMED, ROW_COUNT DESC;

In [ ]:
SELECT
    {{C_VAL}} AS CONDITION_VALUE,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_SNOMED}}) AS UNIQUE_SNOMED_CODES,
    LISTAGG(DISTINCT {{C_SNOMED}}::STRING, ' | ') AS SNOMED_CODES
FROM {{T}}
WHERE {{C_VAL}} IS NOT NULL
GROUP BY 1
HAVING COUNT(DISTINCT {{C_SNOMED}}) > 1
ORDER BY UNIQUE_SNOMED_CODES DESC, ROW_COUNT DESC;

In [ ]:
SELECT
    {{C_SNOMED2}} AS SECONDARY_SNOMED,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_VAL}}) AS UNIQUE_NAMES,
    LISTAGG(DISTINCT {{C_VAL}}::STRING, ' | ') AS NAME_VARIATIONS,
    COUNT(DISTINCT {{C_SNOMED}}) AS DISTINCT_PRIMARY_SNOMED_CODES,
    LISTAGG(DISTINCT {{C_SNOMED}}::STRING, ' | ') AS PRIMARY_SNOMED_CODES,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM {{T}}
WHERE {{C_SNOMED2}} IS NOT NULL
  AND TRIM({{C_SNOMED2}}::STRING) <> ''
GROUP BY 1
ORDER BY ROW_COUNT DESC, SECONDARY_SNOMED;

In [ ]:
SELECT
    {{C_SNOMED}} AS SNOMED,
    {{C_SNOMED2}} AS SECONDARY_SNOMED,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_VAL}}) AS UNIQUE_NAMES,
    LISTAGG(DISTINCT {{C_VAL}}::STRING, ' | ') AS NAME_VARIATIONS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM {{T}}
GROUP BY 1, 2
ORDER BY ROW_COUNT DESC;

## 10. Date distribution

`Date` is when the history was **recorded**. Recency is today minus that date, so values stay positive; anything after today is flagged as future.

In [ ]:
SELECT
    CURRENT_DATE() AS TODAY,
    MIN({{C_DATE}}) AS MIN_DATE,
    MAX({{C_DATE}}) AS MAX_DATE,
    DATEDIFF('day', MIN({{C_DATE}})::DATE, MAX({{C_DATE}})::DATE) AS SPAN_DAYS,
    SUM(IFF({{C_DATE}}::DATE > CURRENT_DATE(), 1, 0)) AS FUTURE_ROWS,
    SUM(IFF({{C_DATE}} IS NULL, 1, 0)) AS MISSING_DATE_ROWS
FROM {{T}};

In [ ]:
SELECT
    YEAR({{C_DATE}}) AS RECORD_YEAR,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_VAL}}) AS UNIQUE_CONDITIONS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM {{T}}
GROUP BY 1
ORDER BY 1;

In [ ]:
WITH bucketed AS (
    SELECT
        CASE
            WHEN {{C_DATE}} IS NULL THEN 90
            WHEN {{C_DATE}}::DATE > CURRENT_DATE() THEN 80
            WHEN DATEDIFF('day', {{C_DATE}}::DATE, CURRENT_DATE()) <= 30 THEN 1
            WHEN DATEDIFF('day', {{C_DATE}}::DATE, CURRENT_DATE()) <= 90 THEN 2
            WHEN DATEDIFF('day', {{C_DATE}}::DATE, CURRENT_DATE()) <= 180 THEN 3
            WHEN DATEDIFF('day', {{C_DATE}}::DATE, CURRENT_DATE()) <= 365 THEN 4
            WHEN DATEDIFF('year', {{C_DATE}}::DATE, CURRENT_DATE()) <= 2 THEN 5
            WHEN DATEDIFF('year', {{C_DATE}}::DATE, CURRENT_DATE()) <= 5 THEN 6
            WHEN DATEDIFF('year', {{C_DATE}}::DATE, CURRENT_DATE()) <= 7 THEN 7
            WHEN DATEDIFF('year', {{C_DATE}}::DATE, CURRENT_DATE()) <= 10 THEN 8
            ELSE 9
        END AS SORT_ORDER,
        {{C_PT}} AS PATIENT_ID
    FROM {{T}}
)
SELECT
    SORT_ORDER,
    CASE SORT_ORDER
        WHEN 1 THEN '0-30 days ago'
        WHEN 2 THEN '31-90 days ago'
        WHEN 3 THEN '91-180 days ago'
        WHEN 4 THEN '181-365 days ago'
        WHEN 5 THEN '1-2 years ago'
        WHEN 6 THEN '3-5 years ago'
        WHEN 7 THEN '6-7 years ago'
        WHEN 8 THEN '8-10 years ago'
        WHEN 9 THEN 'More than 10 years ago'
        WHEN 80 THEN 'Future (after today)'
        ELSE 'Unknown (missing date)'
    END AS DATE_RANGE,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT PATIENT_ID) AS UNIQUE_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM bucketed
GROUP BY 1, 2
ORDER BY SORT_ORDER;

## 11. Conditions per patient, and one patient drill-down

The drill-down auto-picks the patient with the most rows. To inspect a specific patient, replace the subquery in the `WHERE` clause with that id.

In [ ]:
WITH per_patient AS (
    SELECT
        {{C_PT}} AS PATIENT_ID,
        COUNT(*) AS ROW_COUNT
    FROM {{T}}
    WHERE {{C_PT}} IS NOT NULL
    GROUP BY 1
)
SELECT
    ROW_COUNT AS CONDITIONS_PER_PATIENT,
    COUNT(*) AS NUMBER_OF_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_PATIENTS
FROM per_patient
GROUP BY 1
ORDER BY 1;

In [ ]:
SELECT
    {{C_PT}} AS PATIENT_ID,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_VAL}}) AS UNIQUE_CONDITIONS,
    COUNT(DISTINCT {{C_SNOMED}}) AS UNIQUE_SNOMED_CODES,
    MIN({{C_DATE}}) AS FIRST_DATE,
    MAX({{C_DATE}}) AS LAST_DATE
FROM {{T}}
WHERE {{C_PT}} IS NOT NULL
GROUP BY 1
ORDER BY ROW_COUNT DESC
LIMIT 25;

In [ ]:
SELECT
    {{C_PT}} AS PATIENT_ID,
    {{C_ID}} AS MEDICAL_HISTORY_ID,
    {{C_ENC}} AS ENCOUNTER_ID,
    {{C_SRC}} AS SOURCE_CATEGORY,
    {{C_VAL}} AS CONDITION_VALUE,
    {{C_SNOMED}} AS SNOMED,
    {{C_SNOMED2}} AS SECONDARY_SNOMED,
    {{C_DATE}} AS RECORD_DATE
FROM {{T}}
WHERE {{C_PT}} = (
        SELECT {{C_PT}}
        FROM {{T}}
        WHERE {{C_PT}} IS NOT NULL
        GROUP BY 1
        ORDER BY COUNT(*) DESC
        LIMIT 1
      )
ORDER BY {{C_DATE}} NULLS LAST, {{C_ID}};

## Notes

- **Downloading:** run a SQL cell, then use the download arrow on that result grid. Chart and Pivot are on the same toolbar.
- If a cell fails with **invalid identifier**, copy names from `describe_table` into `COL` in the config cell. `Source/Category` and `Secondary SNOMED` need `QUOTE_COLUMNS = True`.
- If the table is missing, use `find_table`, then change only `TABLE_NAME`.
- Jinja braces only substitute **string** variables from an earlier Python cell, so run `config` before the SQL cells.